# VINS AI VIDEO STUDIO — Cloud Renderer
Run cells from top to bottom in Google Colab. The free runtime is temporary; download your MP4 before closing it.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core
!pip -q install edge-tts pillow

In [ ]:
import os, json, zipfile, shutil, subprocess, glob, math, textwrap, re, sys
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageFilter
from IPython.display import display, FileLink
from google.colab import files

print("Upload the project ZIP downloaded from VINS AI Video Studio")
uploaded = files.upload()
zip_name = next(iter(uploaded))
work = Path("/content/vins_work")
if work.exists(): shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as z: z.extractall(work)
project = json.loads((work/"project.json").read_text(encoding="utf-8"))
assets_dir = work/"assets"
out = Path("/content/vins_output")
out.mkdir(exist_ok=True)
W,H = map(int, project.get("format","1080x1920").split("x"))
brand = project.get("brand","VINS ENTERPRISES")
contact = project.get("contact","")
scenes = project.get("scenes",[])
if not scenes: raise ValueError("No scenes found in project ZIP")

# Install/use a Kannada-capable font
font_candidates = [
 "/usr/share/fonts/truetype/noto/NotoSansKannada-Regular.ttf",
 "/usr/share/fonts/truetype/noto/NotoSansKannadaUI-Regular.ttf",
 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
def font(size):
    for p in font_candidates:
        if os.path.exists(p): return ImageFont.truetype(p,size)
    return ImageFont.load_default()

# Pick uploaded images in upload order
image_paths = [work/p for p in project.get("assets",{}).get("images",[]) if (work/p).exists()]
logo_rel = project.get("assets",{}).get("logo")
logo_path = work/logo_rel if logo_rel and (work/logo_rel).exists() else None
music_rel = project.get("assets",{}).get("music")
music_path = work/music_rel if music_rel and (work/music_rel).exists() else None

def cover(img, size):
    img=ImageOps.exif_transpose(img).convert("RGB")
    return ImageOps.fit(img,size,method=Image.Resampling.LANCZOS,centering=(0.5,0.5))

def wrap_text(draw, text, fnt, max_width):
    words=text.split()
    lines=[]; line=""
    for word in words:
        test=(line+" "+word).strip()
        if draw.textbbox((0,0),test,font=fnt)[2] <= max_width: line=test
        else:
            if line: lines.append(line)
            line=word
    if line: lines.append(line)
    return lines

slide_files=[]
for i, scene in enumerate(scenes):
    bg=Image.new("RGB",(W,H),(13,20,42))
    # Select a photo for each scene, cycling when fewer photos are supplied
    if image_paths:
        try:
            photo=Image.open(image_paths[i % len(image_paths)])
            photo=cover(photo,(W,H)).filter(ImageFilter.GaussianBlur(1))
            bg=Image.blend(bg,photo,0.58)
        except Exception as e: print("Skipping image:",e)
    draw=ImageDraw.Draw(bg,"RGBA")
    # dark lower gradient panel
    panel_top=int(H*0.48)
    draw.rectangle((0,panel_top,W,H),fill=(5,10,25,205))
    draw.rectangle((0,0,W,int(H*.12)),fill=(6,10,24,175))
    title=scene.get("title",f"SCENE {i+1}")
    draw.text((int(W*.07),int(H*.045)),brand,font=font(max(22,int(W*.035))),fill=(255,255,255,255))
    draw.text((int(W*.07),int(H*.53)),title,font=font(max(28,int(W*.055))),fill=(196,181,253,255))
    narration=scene.get("narration","")
    lines=wrap_text(draw,narration,font(max(22,int(W*.038))),int(W*.84))
    y=int(H*.62); line_h=int(H*.052)
    for line in lines[:7]:
        draw.text((int(W*.07),y),line,font=font(max(22,int(W*.038))),fill=(255,255,255,255),stroke_width=1,stroke_fill=(0,0,0,180))
        y+=line_h
    # footer
    draw.rectangle((0,int(H*.91),W,H),fill=(8,12,28,220))
    footer=contact[:120]
    draw.text((int(W*.05),int(H*.935)),footer,font=font(max(14,int(W*.022))),fill=(255,255,255,255))
    if logo_path:
        try:
            lg=Image.open(logo_path).convert("RGBA"); lg.thumbnail((int(W*.20),int(H*.12)))
            bg.paste(lg,(W-lg.width-int(W*.05),int(H*.14)),lg)
        except: pass
    p=out/f"slide_{i:03d}.png"; bg.save(p); slide_files.append(p)

# Narration with Kannada neural voice; online service may be unavailable or throttled
script=project.get("script","")
voice=project.get("voice","kn-IN-GaganNeural")
audio=out/"voiceover.mp3"
try:
    import edge_tts, asyncio
    async def make_voice():
        communicate=edge_tts.Communicate(script,voice)
        await communicate.save(str(audio))
    asyncio.run(make_voice())
    print("Voice-over generated:",voice)
except Exception as e:
    print("Voice generation unavailable:",e)
    print("Upload your own narration audio as a fallback.")
    alt=files.upload()
    if alt:
        fn=next(iter(alt)); shutil.copy(fn,audio)

def probe_duration(path):
    r=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",str(path)],capture_output=True,text=True)
    return float(r.stdout.strip())
if not audio.exists(): raise RuntimeError("No narration audio was generated or uploaded.")
total=probe_duration(audio)
weights=[max(1,len(s.get("narration","").split())) for s in scenes]
sumw=sum(weights)
durations=[max(2.5,total*w/sumw) for w in weights]
# Normalize durations to exactly audio duration
scale=total/sum(durations); durations=[d*scale for d in durations]

# Build still-image scene clips with a slow zoom; encode each and concatenate
clips=[]
for i,(slide,dur) in enumerate(zip(slide_files,durations)):
    clip=out/f"clip_{i:03d}.mp4"
    frames=max(1,int(dur*25))
    vf=f"scale={W}:{H},zoompan=z='min(zoom+0.00035,1.08)':d={frames}:s={W}x{H}:fps=25,format=yuv420p"
    subprocess.run(["ffmpeg","-y","-loop","1","-i",str(slide),"-t",str(dur),"-vf",vf,"-r","25","-c:v","libx264","-preset","ultrafast","-pix_fmt","yuv420p",str(clip)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    clips.append(clip)
concat=out/"clips.txt"
concat.write_text("".join(f"file '{p.as_posix()}'\n" for p in clips))
silent=out/"silent.mp4"
subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",str(concat),"-c","copy",str(silent)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)

# Generate approximate scene-aligned SRT subtitles
srt=[]; cursor=0
def stamp(sec):
    ms=int(round((sec-int(sec))*1000)); whole=int(sec)
    return f"{whole//3600:02}:{(whole%3600)//60:02}:{whole%60:02},{ms:03}"
for sc,d in zip(scenes,durations):
    start=cursor; end=cursor+d
    srt.append(f"{len(srt)+1}\n{stamp(start)} --> {stamp(end)}\n{sc.get('narration','')}\n")
    cursor=end
srt_path=out/"VINS_Subtitles.srt"
srt_path.write_text("\n".join(srt),encoding="utf-8")

# Mix narration and optional music, then mux to MP4
final=out/"VINS_Promo.mp4"
if music_path:
    subprocess.run(["ffmpeg","-y","-i",str(silent),"-i",str(audio),"-stream_loop","-1","-i",str(music_path),"-filter_complex","[1:a]volume=1.0[voice];[2:a]volume=0.12[music];[voice][music]amix=inputs=2:duration=first:dropout_transition=2[a]","-map","0:v","-map","[a]","-c:v","copy","-c:a","aac","-shortest",str(final)],check=True)
else:
    subprocess.run(["ffmpeg","-y","-i",str(silent),"-i",str(audio),"-map","0:v","-map","1:a","-c:v","copy","-c:a","aac","-shortest",str(final)],check=True)
print("Rendering complete:",final)
display(FileLink(str(final)))
display(FileLink(str(srt_path)))
